In [ ]:
targetSku = '' 
capacityName = ''
subscriptionId = '' 
resourceGroup = ''

##### **Assign service principal credential information**
Update the **keyVaultEndpoint** to the Azure Key Vault url and the secret name values if the service principal credentials are stored there.\
These credential information can be hard coded for testing purposes and to get started.

In [ ]:
keyVaultEndpoint = 'https://<key-vault-name>.vault.azure.net/'

tenantId = notebookutils.credentials.getSecret(keyVaultEndpoint, '<secret-name>')
clientId = notebookutils.credentials.getSecret(keyVaultEndpoint, '<secret-name>')
secret = notebookutils.credentials.getSecret(keyVaultEndpoint, '<secret-name>')

##### **Acquire Tokens and create the API headers**
We need to acquire two tokens:
- PBI audience so that we're able to use the PBI/Fabric APIs.
- Azure Management audience to scale the capacity within Azure.

In [ ]:
from azure.identity import ClientSecretCredential

api_pbi = 'https://analysis.windows.net/powerbi/api/.default'
api_azuremgmt = 'https://management.core.windows.net/.default'

auth = ClientSecretCredential(tenant_id=tenantId, client_id=clientId, client_secret=secret)
header_pbi = {'Authorization': f'Bearer {auth.get_token(api_pbi).token}', 'Content-type': 'application/json'}
header_azuremgmt = {'Authorization': f'Bearer {auth.get_token(api_azuremgmt).token}', 'Content-type': 'application/json'}

##### **Perform the scaling operation within Azure**

In [ ]:
import requests, json, time

In [ ]:
def get_fabric_capacity_sku(subscription_id, capacity_name, headers):
    """
    Get the current SKU for a Microsoft Fabric capacity.
    
    Args:
        subscription_id: Azure subscription ID
        capacity_name: Name of the Fabric capacity
        headers: Authorization headers for the API
    
    Returns:
        dict with SKU info, or None if not found
    """
    url = f'https://management.azure.com/subscriptions/{subscription_id}/providers/Microsoft.Fabric/capacities/?api-version=2022-07-01-preview'
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"API Error: {response.status_code} - {response.text}")
        return None
    
    data = response.json()
    
    for capacity in data["value"]:
        if capacity["name"] == capacity_name:
            return {
                "name": capacity_name,
                "skuName": capacity["sku"]["name"],
                "skuTier": capacity["sku"]["tier"],
                "provisioningState": capacity["properties"]["provisioningState"],
                "state": capacity["properties"]["state"]
            }
    
    print(f"Capacity '{capacity_name}' not found")
    return None

In [ ]:
def poll_fabric_capacity(subscription_id, capacity_name, headers, 
                         target_state="Succeeded", 
                         poll_interval=10, 
                         timeout=600):
    """
    Poll the Fabric capacity API until provisioningState reaches target state.
    
    Args:
        subscription_id: Azure subscription ID
        capacity_name: Name of the Fabric capacity to monitor
        headers: Authorization headers for the API
        target_state: Desired provisioningState (default: "Succeeded")
        poll_interval: Seconds between polls (default: 30)
        timeout: Max seconds to wait (default: 600 = 10 minutes)
    
    Returns:
        dict with final capacity info, or None if not found/timeout
    """
    url = f'https://management.azure.com/subscriptions/{subscription_id}/providers/Microsoft.Fabric/capacities/?api-version=2022-07-01-preview'
    
    start_time = time.time()
    attempt = 0
    
    while True:
        attempt += 1
        elapsed = time.time() - start_time
        
        if elapsed > timeout:
            print(f"Timeout after {timeout} seconds")
            return None
        
        response = requests.get(url, headers=header_azuremgmt)
        
        if response.status_code != 200:
            print(f"API Error: {response.status_code} - {response.text}")
            time.sleep(poll_interval)
            continue
        
        data = response.json()
        
        for capacity in data["value"]:
            if capacity["name"] == capacity_name:
                provisioning_state = capacity["properties"]["provisioningState"]
                sku_name = capacity["sku"]["name"]
                state = capacity["properties"]["state"]
                
                print(f"[Attempt {attempt}] {capacity_name}: provisioningState={provisioning_state}, state={state}, sku={sku_name}")
                
                if provisioning_state == target_state:
                    print(f"✓ Reached target state '{target_state}' after {elapsed:.1f} seconds")
                    return {
                        "name": capacity_name,
                        "provisioningState": provisioning_state,
                        "state": state,
                        "skuName": sku_name
                    }
                
                # Check for terminal failure states
                if provisioning_state in ["Failed", "Canceled"]:
                    print(f"✗ Reached terminal state '{provisioning_state}'")
                    return None
                
                break
        else:
            print(f"[Attempt {attempt}] Capacity '{capacity_name}' not found")
        
        print(f"  Waiting {poll_interval} seconds before next poll...")
        time.sleep(poll_interval)


In [ ]:
# Check current SKU before scaling
current = get_fabric_capacity_sku(subscriptionId, capacityName, header_azuremgmt)
currentSku = current['skuName']

if current:
    print(f"Current SKU: {currentSku}")
    
    if currentSku != targetSku:
        print(f'\nScaling from {currentSku} to {targetSku}')

        url = f'https://management.azure.com/subscriptions/{subscriptionId}/resourceGroups/{resourceGroup}/providers/Microsoft.Fabric/capacities/{capacityName}?api-version=2022-07-01-preview'
        body = {"sku": {"name": f"{targetSku}", "tier": "Fabric"}}
        
        response = requests.patch(url, headers=header_azuremgmt, data=json.dumps(body))
        print(response, response.text)
    else:
        print("Already at target SKU, no scaling needed")

In [ ]:
result = poll_fabric_capacity(
    subscription_id=subscriptionId,
    capacity_name=capacityName,
    headers=header_azuremgmt,
    target_state="Succeeded",
    poll_interval=10,  # Check every 10 seconds
    timeout=600        # Give up after 10 minutes
)

if result:
    print(f"\nFinal result: {result}")